# Knowledge graph using Neo 4J

## Understanding Indian bail judgements

> Data source from Kaggle  
> DOI: **willian oliveira. (2026). Indian-Bail-Judgments [Dataset]. Kaggle. https://doi.org/10.34740/KAGGLE/DSV/19614939**

In [65]:
import pandas
import re
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase, Session, Record, ManagedTransaction
import atexit
from IPython import get_ipython

In [66]:
# Load from environment
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('KG_DATABASE_URL')
NEO4J_USERNAME = os.getenv('KG_DATABASE_USER')
NEO4J_PASSWORD = os.getenv('KG_DATABASE_PASSWORD')
NEO4J_DATABASE = os.getenv('KG_DATABASE') or 'neo4j'

In [67]:
NEO4J_AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

# 1. Initiate session
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
session = driver.session(database=NEO4J_DATABASE)

# 2. Define the unified cleanup logic
def close_neo4j_driver():
  global driver
  global session
  if session is not None:
    print("\n[Neo4j Cleanup] Closing active session...")
    try:
      session.close()
      print("[Neo4j Cleanup] session closed successfully.")
    except Exception as e:
      print(f"[Neo4j Cleanup] Error closing session: {e}")
    finally:
      session = None
  if driver is not None:
    print("\n[Neo4j Cleanup] Closing active connection pool...")
    try:
      driver.close()
      print("[Neo4j Cleanup] Driver shut down successfully.")
    except Exception as e:
      print(f"[Neo4j Cleanup] Error closing driver: {e}")
    finally:
      driver = None


# 3. Hook 1: Handle standard Python exit (Kernel > Shut Down / Restart)
atexit.register(close_neo4j_driver)

# 4. Hook 2: Jupyter/IPython explicit magic hook (Optional extra layer)
ip = get_ipython()
if ip is not None:

  def ipython_cleanup_hook(ipython):
    close_neo4j_driver()

  # Fires right before the kernel wraps up its event loop
  ip.events.register("post_execute", lambda: None)  # Dummy to ensure registration context

In [68]:
# Importing data
dataframe = pandas.read_csv("../data/Indian-Bail-Judgments new.csv")

## Understanding Data

In [69]:
# Understanding data
dataframe['case_id'] = dataframe['case_id'].astype(str)
dataframe.describe(include='all')

,case_id,case_title,court,date,judge,bail_type,bail_cancellation_case,landmark_case,accused_name,accused_gender,...,bail_outcome,bail_outcome_label_detailed,crime_type,facts,legal_issues,judgment_reason,summary,bias_flag,parity_argument_used,region
count,1200,1200,1200,1200,1200,1200,1200,1200,1200,1200,...,1200,1200,1200,1200,1200,1200,1200,1200,1200,1200
unique,1200,1182,78,884,573,6,2,2,1157,4,...,2,1172,12,1198,1195,1197,1198,2,2,28
top,923,Sumit Kumar vs State Of Delhi,Punjab and Haryana High Court,2024-12-11,Justice Rajesh Bhardwaj,Regular,False,False,Not specified,Male,...,Granted,Regular bail granted after compliance with int...,Narcotics,Ram Niwas Gupta was arrested while attempting ...,Whether preventive detention is valid when the...,The Delhi High Court quashed the detention cit...,Delhi High Court quashed COFEPOSA detention of...,False,False,Uttar Pradesh
freq,1,3,130,32,34,906,1084,1053,11,1071,...,736,18,134,2,2,2,2,1187,859,130


In [70]:
dataframe.columns

Index(['case_id', 'case_title', 'court', 'date', 'judge', 'bail_type',
       'bail_cancellation_case', 'landmark_case', 'accused_name',
       'accused_gender', 'prior_cases', 'bail_outcome',
       'bail_outcome_label_detailed', 'crime_type', 'facts', 'legal_issues',
       'judgment_reason', 'summary', 'bias_flag', 'parity_argument_used',
       'region'],
      dtype='str')

In [71]:
# Understanding data
dataframe.accused_gender.unique()

<StringArray>
['Male', 'Multiple', 'Female', 'Unknown']
Length: 4, dtype: str

## Parsing data

In [72]:
# Parsing data
case_dict = dataframe.to_dict(orient='records')
type(case_dict)

list

In [73]:
def parse_judge(case: dict, unique_judge_list: list = []) -> None:
    '''
    Parse the judge field in a case dictionary and update the unique judge list.

    Parameters:
    case (dict): A dictionary representing a case, which may contain a 'judge' field.
    unique_judge_list (list): A list to store unique judge names across cases.
    '''
    judge_list = []
    case['judge_list'] = judge_list
    if 'judge' in case:
        judges = re.split(r",|( and )", case['judge'])
        for judge in judges:
            if judge is None:
                continue
            judge = judge.strip()
            if judge.lower() in ['and', 'not present', ',']:
                continue
            judge_list.append(judge)
            if judge not in unique_judge_list:
                unique_judge_list.append(judge)

In [74]:
def parse_accused(case: dict, unique_accused_list: list = []) -> None:
    '''
    Parse the accused field in a case dictionary and update the unique accused list.

    Parameters:
    case (dict): A dictionary representing a case, which may contain an 'accused_name' field.
    unique_accused_list (list): A list to store unique accused names across cases.
    '''
    accused_list = []
    case['accused_list'] = accused_list
    if 'accused_name' in case:
        inp_accused = re.split(r",|( and )|( & )", case['accused_name'])
        for accused in inp_accused:
            if accused is None:
                continue
            accused = accused.strip()
            if accused.lower() in ['and', 'not present', ',', '']:
                continue
            if accused.lower() == 'unknown':
                continue
            accused_list.append(accused)
            if accused not in unique_accused_list:
                unique_accused_list.append(accused)

In [75]:
def insert_case(case: dict, neo4j_session: Session | None = None) -> Record | None:
    """
    Insert a case into the Neo4j database.
    
    Parameters:
    case (dict): A dictionary representing a case, which needs to be saved into the database.
    neo4j_session (Session, optional): An existing Neo4j session. If not provided, the global session will be used.
    """
    if neo4j_session is None:
        neo4j_session = globals().get("session")

    if neo4j_session is None:
        raise RuntimeError("Neo4j session is not initialized.")

    result = neo4j_session.execute_write(
        lambda tx: tx.run(
            """
            MERGE (c:Case {id: $id})
            SET c.title = $title,
                c.date = $date,
                c.bail_cancellation_case = $bail_cancellation_case,
                c.landmark_case = $landmark_case,
                c.crime_type = $crime_type,
                c.facts = $facts,
                c.legal_issues = $legal_issues,
                c.judgment_reason = $judgment_reason,
                c.summary = $summary,
                c.bias_flag = $bias_flag,
                c.parity_argument_used = $parity_argument_used,
                c.region = $region,
                c.had_other_accused = $had_other_accused,
                c.accused_count = $accused_count
            RETURN c AS case
            """,
            id=case.get("case_id"),
            title=case.get("case_title"),
            date=case.get("date"),
            bail_cancellation_case=case.get("bail_cancellation_case"),
            landmark_case=case.get("landmark_case"),
            crime_type=case.get("crime_type"),
            facts=case.get("facts"),
            legal_issues=case.get("legal_issues"),
            judgment_reason=case.get("judgment_reason"),
            summary=case.get("summary"),
            bias_flag=case.get("bias_flag"),
            parity_argument_used=case.get("parity_argument_used"),
            region=case.get("region"),
            had_other_accused=case.get("had_other_accused"),
            accused_count=case.get("accused_count"),
        ).single(strict=False)
    )

    return result

In [76]:
def insert_accused_helper(accused: str, case_id: str | None, case: dict, tx: ManagedTransaction) -> Record | None:
    result = tx.run(
        """
        MERGE (p:Person {name: $name})
        SET p.name = $name,
        p.type = 'Accused',
        p.gender = $gender,
        p.prior_cases = $prior_cases
        RETURN p AS person
        """,
        name=accused,
        gender=case.get("accused_gender") if case["accused_gender"] in ("Male", "Female", "Other") else None,
        prior_cases=case.get("prior_cases")
    )
    if case_id:
        tx.run(
            """
            MERGE (a:Person {name: $name})
            WITH a
            MATCH (c:Case {id: $case_id})
            MERGE (a)-[:ACCUSED_IN]->(c)
        RETURN a AS accused
        """,
        name=accused,
        case_id=case_id
    )
    return result.single(strict=False)

def insert_accused(case: dict, case_entity: Record | None, neo4j_session: Session | None = None) -> Record | None:
    """
    Insert an accused into the Neo4j database.
    
    Parameters:
    accused_name (str): The name of the accused to be saved into the database.
    neo4j_session (Session, optional): An existing Neo4j session. If not provided, the global session will be used.
    """
    if neo4j_session is None:
        neo4j_session = globals().get("session")

    if neo4j_session is None:
        raise RuntimeError("Neo4j session is not initialized.")

    result = None

    for accused in case['accused_list']:
        result = neo4j_session.execute_write(
            lambda tx: insert_accused_helper(accused, case_entity['case']['id'] if case_entity else None, case, tx)
        )

    return result

In [77]:
unique_judge_list = []
unique_accused_list = []
case_entity = None
for case in case_dict:
    parse_judge(case, unique_judge_list)
    parse_accused(case, unique_accused_list)
    case_entity = insert_case(case)
    accused_entity = insert_accused(case, case_entity)
    # insert_judge(case, case_entity)
    # insert_bail(case, case_entity)
    # insert_court(case, case_entity)

In [78]:
print(case_entity)

<Record case=<Node element_id='4:5cf83fa5-f705-45f3-a4a0-e9496adcc0ae:2461' labels=frozenset({'Case'}) properties={'date': '2025-06-02', 'summary': 'Kantilal granted bail in school robbery attempt case due to simple injuries, no prior record, and completed investigation.', 'bias_flag': False, 'judgment_reason': 'Court considered the minor nature of injuries, lack of recovery from the accused, absence of criminal history, and filed chargesheet to grant bail.', 'crime_type': 'Theft or Robbery', 'title': 'Kantilal vs State Of Rajasthan', 'facts': 'Kantilal was arrested in connection with an alleged attempted robbery at a school in Bhilwara, Rajasthan. The prosecution claimed he and others planned the act and assaulted police during intervention. The defense argued he had no role in causing injuries and no criminal record. Injuries to victims were minor and caused by blunt weapons.', 'legal_issues': 'Whether bail can be granted in a planned robbery attempt case when the accused has no prio

In [79]:
close_neo4j_driver()


[Neo4j Cleanup] Closing active session...
[Neo4j Cleanup] session closed successfully.

[Neo4j Cleanup] Closing active connection pool...
[Neo4j Cleanup] Driver shut down successfully.
